In [31]:
macro test(expr)
    print(expr.head)
    if expr.head == :braces
        return Expr(:vect, expr.args...)
    end
    return expr
end

@test (macro with 1 method)

In [33]:
@test {1, 2}

braces

2-element Vector{Int64}:
 1
 2

In [38]:
typeof(:{})

Expr

In [41]:
macro print_tokens(expr)
    # Ensure we are dealing with an expression
    if !isa(expr, Expr)
        return :(println("Not an expression: ", typeof($expr)))
    end

    # Loop through the arguments of the expression
    for (i, arg) in enumerate(expr.args)
        if isa(arg, Expr)
            println("Arg $i ($arg) is an Expr, Head: ", arg.head)
        else
            println("Arg $i ($arg) is a literal/symbol: ", typeof(arg))
        end
    end
    
    # Return the original expression to be executed
    # return esc(expr)
end

# Usage
@print_tokens(a = b + 5)
# Output:
# Arg 1 is a literal/symbol: Symbol
# Arg 2 is an Expr, Head: call


Arg 1 (a) is a literal/symbol: Symbol
Arg 2 (b + 5) is an Expr, Head: call


In [35]:
@print_tokens {Intersect(:c, :f)}

Arg 1 is an Expr, Head: call


LoadError: syntax: { } vector syntax is discontinued around In[35]:1

In [52]:
# Helper function to print types recursively
function print_expr_types(ex, depth=0)
    indent = "  " ^ depth
    if isa(ex, Expr)
        println(indent, "Expr (Head: ", ex.head, ")")
        for arg in ex.args
            print_expr_types(arg, depth + 1)
        end
    else
        println(indent, typeof(ex), ": ", ex)
    end
end

# The macro calls the function on its input expression
macro print_types(expr)
    # quote...end keeps the expression for evaluation/printing
    return quote
        print_expr_types($(esc(expr)))
    end
end

# Usage:
@print_types min(1, 2)
# Output:
# Expr (Head: call)
#   Symbol: +
#   Int64: 1
#   Expr (Head: call)
#     Symbol: *
#     Int64: 2
#     Symbol: x


Int64: 1


In [59]:
macro print_types(expr)
    # Define a recursive function inside the macro
    function walk_and_print(ex, depth=0)
        indent = "  " ^ depth
        if ex isa Expr
            println(indent, "Expr Head: ", ex.head)
            for arg in ex.args
                walk_and_print(arg, depth + 1)
            end
        elseif ex isa Symbol
            println(indent, "Symbol: ", ex)
        elseif ex isa LineNumberNode
            # Skip for cleaner output
        else
            println(indent, "Literal: ", typeof(ex), " = ", ex)
        end
    end

    # Return the original expression to allow normal execution
    println("--- Structure Begin ---")
    walk_and_print(expr)
    println("--- Structure End ---")
    # return esc(expr)
end

# Usage:
@print_types l1 = {Intersect(a, b)}


--- Structure Begin ---
Expr Head: =
  Symbol: l1
  Expr Head: braces
    Expr Head: call
      Symbol: Intersect
      Symbol: a
      Symbol: b
--- Structure End ---
